# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python data tools, in line with FAIR data principles.

### Dataset Source
This dataset is described by a Croissant schema accessible at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

*Cite as: Liu, Y, Duan, X, Yang, S, Zhang, Y and Han, S 2026 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Frontiers*

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview

Let’s review the available record sets and their fields in the FAIR² dataset. All references will use the entities' `@id` fields.

In [ ]:
# Show all record set @ids and their contained fields (by @id)
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id} | Name: {getattr(rs, 'name', '<unnamed>')}")
    print("  Fields:")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', '<unnamed>')} | dataType: {getattr(field, 'data_type', '<unknown>')}")
    else:
        print("    <No fields found>")
    print()

## 3. Data Extraction

Load records from each record set into pandas DataFrames for further analysis. All references are by `@id`.

In [ ]:
# Identify all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"Found record set @ids: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

# Quick preview: show columns for the first populated record set
first_df_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        first_df_id = rs_id
        print(f"\nColumns in first non-empty record set ({rs_id}):\n{df.columns.tolist()}")
        display(df.head())
        break

# If no data loaded, print a warning
if not dataframes:
    print("No dataframes created -- check the dataset record sets and fields.")

## 4. Exploratory Data Analysis (EDA)

Let’s conduct some exploratory analysis. We'll:
- Select a numeric field by its `@id`
- Filter records by value
- Normalize the selected numeric field
- Group by a chosen key attribute (provided it exists in the data)

*Note: Please replace `'<numeric_field_id>'` and `'<group_field_id>'` with concrete @ids after reviewing the fields above.*

In [ ]:
# For demonstration: 
# Let's assume we're working with the main case-level record set, e.g. '@id': 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/RecordSet/CaseData'
# Find a numeric field, e.g. 'Age' or 'Interval_between_cancers'.

# You may need to adjust these IDs to match actual @ids for your case/data
record_set_id = first_df_id  # Use the id found above

# Pick possible field candidates: inspect columns to choose numeric and groupable fields
print(f"Available columns: {dataframes[record_set_id].columns.tolist()}")

# Example: let's suppose 'age' and 'molecular_status' are present and have @ids 'age' and 'molecular_status'.
# If you know the exact @id (as a column label), fill it in (use the print above)

numeric_field = None
group_field = None

# Try to auto-select a likely numeric field
for col in dataframes[record_set_id].columns:
    # Heuristic: fields containing 'age', 'interval', or 'count' are likely numeric
    if any(word in col.lower() for word in ["age", "interval", "count"]):
        numeric_field = col
        break

# Try to auto-select a likely group field
for col in dataframes[record_set_id].columns:
    # Heuristic: fields containing 'sex', 'group', 'status', or 'category' 
    if any(word in col.lower() for word in ["sex", "group", "status", "site"]):
        group_field = col
        break

if numeric_field is None:
    raise ValueError("Could not find a numeric field for demonstration. Please review your DataFrame's columns and set 'numeric_field' manually.")

print(f"Using numeric field: {numeric_field}")
if group_field:
    print(f"Using group field: {group_field}")

# Convert numeric field to numeric dtype (may be string in CSV)
df = dataframes[record_set_id].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter records: e.g., keep only age > 40
threshold = 40
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)}\n")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First rows with normalized {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by category if possible
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'std', 'count'])
    print(f"Grouped statistics by {group_field}:")
    display(grouped_df)

## 5. Visualization

Visualizing the distribution of the selected numeric field, and comparing normalized values across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Plot the distribution of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If a group field is available, compare normalized values between groups
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=f"{numeric_field}_normalized", data=filtered_df, palette="Set2")
    plt.title(f'Normalized {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f"Normalized {numeric_field}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated loading, inspecting, and analyzing the FAIR² colorectal cancer dataset using `mlcroissant` by referencing Croissant entity `@id` values for all data access.
- We explored available record sets and their fields, extracted records into pandas, filtered and normalized numeric data, grouped results, and visualized field distributions.
- For further work, we recommend extending domain-specific analyses (e.g. statistical tests, survival analysis, biomarker stratification) depending on the research question.

#### Notes:
- All data access and plotting is driven by the Croissant entity `@id` fields, not by arbitrary names, ensuring reproducibility and clarity.
- For new fields or different groupings, simply update the referenced `@id` values as shown above.